### Imports

In [10]:
import pandas as pd
import itertools
from itertools import combinations
from tqdm import tqdm 
import csv
import mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_colwidth', None)

### Load CSV

In [2]:
df = pd.read_csv("Food_Inspections_20240215.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267531 entries, 0 to 267530
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    267531 non-null  int64  
 1   DBA Name         267531 non-null  object 
 2   AKA Name         265058 non-null  object 
 3   License #        267513 non-null  float64
 4   Facility Type    262416 non-null  object 
 5   Risk             267450 non-null  object 
 6   Address          267531 non-null  object 
 7   City             267370 non-null  object 
 8   State            267472 non-null  object 
 9   Zip              267482 non-null  float64
 10  Inspection Date  267531 non-null  object 
 11  Inspection Type  267530 non-null  object 
 12  Results          267531 non-null  object 
 13  Violations       194191 non-null  object 
 14  Latitude         266608 non-null  float64
 15  Longitude        266608 non-null  float64
 16  Location         266608 non-null  obje

In [12]:
df.head(1)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2589375,PERFECT BEGINNINGS CHILD DEVELOPMENT CENTER,PERFECT BEGINNINGS CHILD DEVELOPMENT CENTER,2385784.0,Daycare Above and Under 2 Years,Risk 1 (High),1500 W 119TH ST,CHICAGO,IL,60643.0,02/08/2024,License,Pass,"51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: OBSERVED MISSING DRAIN STOPPERS FOR THE 3-COMPARTMENT SINK. INSTRUCTED TO PROVIDE DRAIN STOPPERS FOR ALL THREE BASINS OF 3-COMPARTMENT SINK AND MAINTAIN. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED SMALL GREASE SPILL BENEATH 3-COMPARTMENT SINK IN CORNER. INSTRUCTED TO CLEAN AND SANITIZE AND ARRANGE AREA TO ALLOW FOR FREQUENT CLEANING AND MAINTAIN. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED BASE COVING MISSING ON EXTERIOR WALL IN KITCHEN AREA. INSTRUCTED TO PROVIDE BASE COVING FOR ALL WALLS IN AREAS SUBJECT TO SPLASHING AND/OR MOISTURE AND MAINTAIN. | 56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSERVED VENTILATION HOOD FILTER HEAVILY SOILED WITH CAKED GREASE. INSTRUCTED TO CLEAN OR REPLACE FILTER AND MAINTAIN.",41.677685,-87.65902,"(41.677685065833224, -87.65901954921286)"


### Association Rule

In [7]:
# columns chosen to run association rule on
# all of these are categorical columns
cols_of_interest = ['Facility Type', 'Risk', 'Inspection Type', 'Results']

# could include 'Violations' column too

In [13]:
def mine_association_rules(df, columns):
    print("1. Selecting relevant categorical columns...")
    
    cols_of_interest = columns
    
    # drop rows with empty values in columns of interest
    df_subset = df[cols_of_interest].dropna()
    
    print("2. One-hot encoding the data...")
    # Convert categorical variables into dummy/indicator variables (True/False format)
    
    # Adding prefixes helps us know which column the value came from in the final rules, 
    # eg: Results_Out of Business ('Out of Business' is a value of 'Results' column)
    df_encoded = pd.get_dummies(df_subset, prefix=cols_of_interest)
    
    # mlxtend requires the dataframe to be strictly boolean type
    df_encoded = df_encoded.astype('bool')
    
    print("3. Running Apriori algorithm to find frequent itemsets...")
    # min_support=0.05 means a combination must appear in at least 5% of all inspections. 
    frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)
    
    if frequent_itemsets.empty:
        print("No frequent itemsets found. Try lowering the min_support.")
        return None

    print("4. Generating association rules...")
    # We use 'lift' as the metric. A lift > 1 means the items are positively correlated.
    # min_threshold=1.2 ensures we only get rules showing a reasonably strong correlation.
    
    # Lift: The strength of the correlation. A lift of 1 means they are independent. 
    # A lift of 2.0 means X and Y appear together twice as often as you would expect by random chance.
    
    # could use support and / or confidence minimum thesholds too
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
    
    rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)
    return rules

rules_df = mine_association_rules(df, cols_of_interest)

# Display the top n strongest correlations
if rules_df is not None:
    # antecedents = LHS, consequents = RHS
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    print("\n--- Top 10 Association Rules ---")
    print(rules_df[display_cols].head(50))

1. Selecting relevant categorical columns...
2. One-hot encoding the data...
3. Running Apriori algorithm to find frequent itemsets...
4. Generating association rules...

--- Top 10 Association Rules ---
                                                                              antecedents  \
0                                                               (Inspection Type_Canvass)   
1                                                               (Results_Out of Business)   
2                       (Inspection Type_Canvass Re-Inspection, Facility Type_Restaurant)   
3                                                      (Risk_Risk 1 (High), Results_Pass)   
4                                                 (Inspection Type_Canvass Re-Inspection)   
5                            (Risk_Risk 1 (High), Facility Type_Restaurant, Results_Pass)   
6                             (Risk_Risk 1 (High), Inspection Type_Canvass Re-Inspection)   
7                                                (Fa

In [14]:
rules_df.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Inspection Type_Canvass),(Results_Out of Business),0.513468,0.070481,0.069281,0.134927,1.914369,1.0,0.033091,1.074498,0.981713,0.134612,0.069332,0.558947
1,(Results_Out of Business),(Inspection Type_Canvass),0.070481,0.513468,0.069281,0.982967,1.914369,1.0,0.033091,28.564823,0.513851,0.134612,0.964992,0.558947
2,"(Inspection Type_Canvass Re-Inspection, Facility Type_Restaurant)","(Risk_Risk 1 (High), Results_Pass)",0.076270,0.394328,0.052436,0.687503,1.743482,1.0,0.022361,1.938171,0.461645,0.125396,0.484050,0.410239
3,"(Risk_Risk 1 (High), Results_Pass)","(Inspection Type_Canvass Re-Inspection, Facility Type_Restaurant)",0.394328,0.076270,0.052436,0.132976,1.743482,1.0,0.022361,1.065402,0.704069,0.125396,0.061388,0.410239
4,(Inspection Type_Canvass Re-Inspection),"(Risk_Risk 1 (High), Facility Type_Restaurant, Results_Pass)",0.112189,0.280866,0.052436,0.467389,1.664103,1.0,0.020926,1.350206,0.449505,0.153943,0.259372,0.327042
5,"(Risk_Risk 1 (High), Facility Type_Restaurant, Results_Pass)",(Inspection Type_Canvass Re-Inspection),0.280866,0.112189,0.052436,0.186694,1.664103,1.0,0.020926,1.091608,0.554939,0.153943,0.083920,0.327042
6,"(Risk_Risk 1 (High), Inspection Type_Canvass Re-Inspection)","(Facility Type_Restaurant, Results_Pass)",0.092238,0.348306,0.052436,0.568483,1.632139,1.0,0.020309,1.510241,0.426662,0.135107,0.337854,0.359515
7,"(Facility Type_Restaurant, Results_Pass)","(Risk_Risk 1 (High), Inspection Type_Canvass Re-Inspection)",0.348306,0.092238,0.052436,0.150546,1.632139,1.0,0.020309,1.068641,0.594308,0.135107,0.064232,0.359515
8,(Inspection Type_Canvass Re-Inspection),"(Risk_Risk 1 (High), Results_Pass)",0.112189,0.394328,0.071838,0.640329,1.623850,1.0,0.027599,1.683961,0.432727,0.165267,0.406162,0.411254
9,"(Risk_Risk 1 (High), Results_Pass)",(Inspection Type_Canvass Re-Inspection),0.394328,0.112189,0.071838,0.182178,1.623850,1.0,0.027599,1.085580,0.634302,0.165267,0.078833,0.411254
